In [ ]:
# ---
# # Analysis of Polar Vortex Dynamics: Z3, Wind, Temperature, and Potential Vorticity (PV)
# ---
# 
# This notebook extends the existing O3 analysis framework to visualize polar vortex dynamics.  
# We will generate GIFs for:  
# 1. Z3 (geopotential height) and wind vectors at 10 hPa and 50 hPa.  
# 2. Temperature (T) distributions at 10 hPa and 50 hPa.  
# 3. Potential vorticity (PV) at 10 hPa, 50 hPa, and the 380K isentropic surface.  
# 
# The analysis settings (years, months, latitude range, etc.) are consistent with the O3 code.  
# Outputs include daily PNG frames and composite GIFs.  
# 
# ---
# 
# ## Import Libraries
# 
# We import necessary libraries for data processing, plotting, and GIF creation.  
# Notably, we use `MetPy` for PV calculations.

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import rc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
import imageio.v2 as imageio
from PIL import Image
import metpy.calc as mpcalc
from metpy.units import units

# Set plotting defaults
rc('font', **{'family': 'sans-serif', 'sans-serif': ['Helvetica']})
rc('text', usetex=False)

# ---
# ## Define Utility Functions
# 
# ### 1. Compute Area-Average (Reused from O3 Code)
# 
# Compute the area-weighted average over latitude and longitude.

def compute_area_average(field):
    """
    Compute the area-average of a field over lat and lon.
    
    If the field has a 'lon' dimension with more than one value, first average over lon,
    then compute the cosine-weighted average over lat.
    """
    if 'lon' in field.dims and field.lon.size > 1:
        field_avg = field.mean(dim='lon')
    else:
        field_avg = field
    if 'lat' in field_avg.dims:
        weights = np.cos(np.deg2rad(field_avg.lat))
        field_avg = field_avg.weighted(weights).mean(dim='lat')
    return field_avg

# ### 2. Load Experiment Fields (Modified for Z3, U, V, T, and PV)
# 
# Load Z3, U, V, T, and other variables from NetCDF files.  
# Select specific pressure levels (10 hPa and 50 hPa) or prepare data for PV calculation.

def load_experiment_field(file_pattern, variables, p_levels=None, lat_min=60, lat_max=90):
    """
    Load and process variables from NetCDF files for mapping or PV calculation.
    
    Parameters:
        file_pattern (str): Path pattern for NetCDF files.
        variables (list): List of variable names to load (e.g., ['Z3', 'U', 'V', 'T']).
        p_levels (list, optional): Pressure levels (Pa) to select (e.g., [1000, 5000]).
        lat_min, lat_max (float): Latitude bounds (degrees).
    
    Returns:
        dict: Dictionary of processed variables with dimensions (time, lat, lon) or (time, plev, lat, lon).
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    data_dict = {}
    
    for var_name in variables:
        var_data = ds[var_name]
        # Select latitude range
        var_data = var_data.sel(lat=slice(lat_min, lat_max))
        # Select pressure levels if specified
        if p_levels and 'plev' in var_data.dims:
            var_data = var_data.sel(plev=p_levels, method='nearest')
        # Average over ensemble members if applicable
        if 'member' in var_data.dims:
            var_data = var_data.mean(dim='member')
        data_dict[var_name] = var_data
    
    ds.close()  # Close dataset to release memory
    return data_dict

# ### 3. Load Data for Nocouple and Small Perturbation Experiments
# 
# Load data for February and March, similar to O3 code.

def load_year_nocouple_field(year, variables, p_levels=None, lat_min=60, lat_max=90):
    """
    Load February and March nocouple data for a given year.
    """
    file_Feb = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    file_Mar = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    data_Feb = load_experiment_field(file_Feb, variables, p_levels, lat_min, lat_max)
    data_Mar = load_experiment_field(file_Mar, variables, p_levels, lat_min, lat_max)
    return data_Feb, data_Mar

def load_year_small_pert_field(year, variables, p_levels=None, lat_min=60, lat_max=90):
    """
    Load February and March small perturbation data for a given year.
    """
    base_path = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}'
    file_Feb = os.path.join(base_path, 'Feb', f'BWCN.e122.f19_g16.002.{year}-02.*.nc')
    file_Mar = os.path.join(base_path, 'Mar', f'BWCN.e122.f19_g16.002.{year}-03.*.nc')
    data_Feb = load_experiment_field(file_Feb, variables, p_levels, lat_min, lat_max)
    data_Mar = load_experiment_field(file_Mar, variables, p_levels, lat_min, lat_max)
    return data_Feb, data_Mar

# ### 4. Compute Potential Vorticity (PV)
# 
# Compute PV using MetPy for specified pressure levels or isentropic surfaces.

def compute_pv(data_dict, p_levels=None, isentropic_level=None):
    """
    Compute potential vorticity (PV) using MetPy.
    
    Parameters:
        data_dict (dict): Dictionary containing U, V, T, Z3 data.
        p_levels (list, optional): Pressure levels (Pa) for PV calculation.
        isentropic_level (float, optional): Isentropic level (K) for PV calculation.
    
    Returns:
        xarray.DataArray: PV field with dimensions (time, lat, lon) or (time, plev, lat, lon).
    """
    u = data_dict['U'] * units.meter / units.second
    v = data_dict['V'] * units.meter / units.second
    t = data_dict['T'] * units.kelvin
    plev = data_dict['U'].plev * units.pascal if 'plev' in data_dict['U'].dims else None
    
    if isentropic_level:
        # Interpolate to isentropic surface (e.g., 380K)
        pv = mpcalc.isentropic_potential_vorticity(u, v, t, isentropic_level * units.kelvin, plev)
    else:
        # Compute PV on pressure levels
        pv = mpcalc.potential_vorticity_baroclinic(u, v, t, plev)
        if p_levels:
            pv = pv.sel(plev=p_levels, method='nearest')
    
    return pv

# ### 5. Plotting Functions for Z3, Wind, T, and PV
# 
# Plot daily polar maps for Z3 and wind, T, and PV, and save frames as PNG images.

def plot_z3_wind_maps(exp_name, data_z3, data_u, data_v, output_dir, p_level, frame_duration=500):
    """
    Plot Z3 (geopotential height) with wind vectors for a given experiment.
    
    Parameters:
        exp_name (str): Experiment name (e.g., "nocoupl").
        data_z3, data_u, data_v (xarray.DataArray): Z3, U, V data.
        output_dir (str): Directory to save PNG frames.
        p_level (float): Pressure level (Pa) for labeling.
        frame_duration (int): Duration for GIF frames (ms).
    """
    os.makedirs(output_dir, exist_ok=True)
    n_frames = data_z3.time.size
    
    for i in range(n_frames):
        z3_field = data_z3.isel(time=i)
        u_field = data_u.isel(time=i)
        v_field = data_v.isel(time=i)
        
        if 'lon' in z3_field.coords and z3_field.lon.size > 0:
            z3_vals, lons = add_cyclic_point(z3_field.values, coord=z3_field.lon.values)
            u_vals, _ = add_cyclic_point(u_field.values, coord=u_field.lon.values)
            v_vals, _ = add_cyclic_point(v_field.values, coord=v_field.lon.values)
        else:
            print(f"Frame {i+1}: 'lon' coordinate missing, skipping.")
            continue
        
        time_label = z3_field.time.dt.strftime('%Y-%m-%d').item()
        z3_avg = float(compute_area_average(z3_field).compute())
        
        fig, ax = plt.subplots(subplot_kw={'projection': ccrs.NorthPolarStereo()}, figsize=(8, 8))
        ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
        ax.coastlines()
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray')
        gl.xlabels_top = False
        gl.ylabels_right = False
        
        # Plot Z3 as filled contours
        cf = ax.contourf(lons, z3_field.lat, z3_vals, levels=20, cmap='viridis',
                         transform=ccrs.PlateCarree())
        # Plot wind vectors (subset for clarity)
        ax.quiver(lons[::5], z3_field.lat[::5], u_vals[::5, ::5], v_vals[::5, ::5],
                  transform=ccrs.PlateCarree(), scale=20)
        
        cbar = fig.colorbar(cf, ax=ax, orientation='horizontal', fraction=0.05, pad=0.08)
        cbar.set_label(f"Z3 (m) at {p_level/100} hPa", fontsize=12)
        ax.set_title(f"{exp_name} Z3 and Wind (Avg: {z3_avg:.2f} m)\nDate: {time_label}", fontsize=14)
        
        out_fname = os.path.join(output_dir, f"frame_{i+1:03d}.png")
        plt.savefig(out_fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved frame {i+1} to {out_fname}")

def plot_temperature_maps(exp_name, data_t, output_dir, p_level, frame_duration=500):
    """
    Plot temperature (T) distributions for a given experiment.
    """
    os.makedirs(output_dir, exist_ok=True)
    n_frames = data_t.time.size
    
    for i in range(n_frames):
        t_field = data_t.isel(time=i)
        
        if 'lon' in t_field.coords and t_field.lon.size > 0:
            t_vals, lons = add_cyclic_point(t_field.values, coord=t_field.lon.values)
        else:
            print(f"Frame {i+1}: 'lon' coordinate missing, skipping.")
            continue
        
        time_val = np.datetime64(t_field.time.values)
        time_label = np.datetime_as_string(time_val, unit='D')
        t_avg = float(compute_area_average(t_field).compute())
        
        fig, ax = plt.subplots(subplot_kw={'projection': ccrs.NorthPolarStereo()}, figsize=(8, 8))
        ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
        ax.coastlines()
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray')
        gl.xlabels_top = False
        gl.ylabels_right = False
        
        cf = ax.contourf(lons, t_field.lat, t_vals, levels=np.linspace(180, 260, 21), cmap='coolwarm',
                         transform=ccrs.PlateCarree())
        cbar = fig.colorbar(cf, ax=ax, orientation='horizontal', fraction=0.05, pad=0.08)
        cbar.set_label(f"Temperature (K) at {p_level/100} hPa", fontsize=12)
        ax.set_title(f"{exp_name} Temperature (Avg: {t_avg:.2f} K)\nDate: {time_label}", fontsize=14)
        
        out_fname = os.path.join(output_dir, f"frame_{i+1:03d}.png")
        plt.savefig(out_fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved frame {i+1} to {out_fname}")

def plot_pv_maps(exp_name, data_pv, output_dir, p_level=None, isentropic_level=None, frame_duration=500):
    """
    Plot potential vorticity (PV) distributions for a given experiment.
    """
    os.makedirs(output_dir, exist_ok=True)
    n_frames = data_pv.time.size
    
    for i in range(n_frames):
        pv_field = data_pv.isel(time=i)
        
        if 'lon' in pv_field.coords and pv_field.lon.size > 0:
            pv_vals, lons = add_cyclic_point(pv_field.values, coord=pv_field.lon.values)
        else:
            print(f"Frame {i+1}: 'lon' coordinate missing, skipping.")
            continue
        
        time_val = np.datetime64(pv_field.time.values)
        time_label = np.datetime_as_string(time_val, unit='D')
        pv_avg = float(compute_area_average(pv_field).compute())
        
        fig, ax = plt.subplots(subplot_kw={'projection': ccrs.NorthPolarStereo()}, figsize=(8, 8))
        ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
        ax.coastlines()
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray')
        gl.xlabels_top = False
        gl.ylabels_right = False
        
        label = f"PV (PVU) at {p_level/100} hPa" if p_level else f"PV (PVU) at {isentropic_level} K"
        cf = ax.contourf(lons, pv_field.lat, pv_vals * 1e6, levels=20, cmap='viridis',
                         transform=ccrs.PlateCarree())  # Convert PV to PVU
        cbar = fig.colorbar(cf, ax=ax, orientation='horizontal', fraction=0.05, pad=0.08)
        cbar.set_label(label, fontsize=12)
        ax.set_title(f"{exp_name} PV (Avg: {pv_avg*1e6:.2f} PVU)\nDate: {time_label}", fontsize=14)
        
        out_fname = os.path.join(output_dir, f"frame_{i+1:03d}.png")
        plt.savefig(out_fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved frame {i+1} to {out_fname}")

# ### 6. GIF Creation (Reused from O3 Code)
# 
# Create GIFs from PNG frames.

def create_gif_from_directory(image_dir, output_gif, duration=500):
    """
    Read PNG images (sorted by filename) from a directory and combine them into a GIF.
    """
    files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])
    if not files:
        print(f"No PNG images found in {image_dir}!")
        return

    images = []
    for fname in files:
        fpath = os.path.join(image_dir, fname)
        try:
            images.append(imageio.imread(fpath))
        except Exception as e:
            print(f"Error reading {fpath}: {e}")
    imageio.mimsave(output_gif, images, duration=duration/1000)
    print(f"Saved GIF: {output_gif}")

# ---
# ## Main Program
# 
# Process each year and month, load data, compute PV, generate frames, and create GIFs.

if __name__ == "__main__":
    # Base output directory for analysis
    base_output_dir = "/home/weiji/restart_exam/20250221/polar_vortex_analysis"
    os.makedirs(base_output_dir, exist_ok=True)
    
    # Define experiment years and months (consistent with O3 code)
    years = ['0008', '0013', '0014', '0019']
    months = ['Feb', 'Mar']
    
    # Define pressure levels for Z3, T, and PV (10 hPa and 50 hPa, in Pa)
    pressure_levels = [1000, 5000]  # 10 hPa = 1000 Pa, 50 hPa = 5000 Pa
    isentropic_level = 380  # 380K isentropic surface for PV
    
    # Loop over each year and month
    for year in years:
        for month in months:
            print(f"\nProcessing Year: {year}, Month: {month}")
            
            # Load data for nocouple and small perturbation experiments
            variables = ['Z3', 'U', 'V', 'T']
            if month == 'Feb':
                data_nocouple_Feb, _ = load_year_nocouple_field(year, variables, pressure_levels)
                data_small_Feb, _ = load_year_small_pert_field(year, variables, pressure_levels)
                data_nocouple = data_nocouple_Feb
                data_small = data_small_Feb
            elif month == 'Mar':
                _, data_nocouple_Mar = load_year_nocouple_field(year, variables, pressure_levels)
                _, data_small_Mar = load_year_small_pert_field(year, variables, pressure_levels)
                data_nocouple = data_nocouple_Mar
                data_small = data_small_Mar
            else:
                continue
            
            # Process each experiment
            for exp_name, data_dict in [("nocoupl", data_nocouple), ("small", data_small)]:
                # Plot Z3 and wind for each pressure level
                for p_level in pressure_levels:
                    combo_dir = os.path.join(base_output_dir, f"{year}_{month}_{exp_name}_Z3_wind_{p_level/100}hPa")
                    plot_z3_wind_maps(exp_name, data_dict['Z3'].sel(plev=p_level),
                                    data_dict['U'].sel(plev=p_level),
                                    data_dict['V'].sel(plev=p_level),
                                    combo_dir, p_level)
                    gif_fname = os.path.join(base_output_dir, f"Z3_wind_{year}_{month}_{exp_name}_{p_level/100}hPa.gif")
                    create_gif_from_directory(combo_dir, gif_fname)
                
                # Plot temperature for each pressure level
                for p_level in pressure_levels:
                    combo_dir = os.path.join(base_output_dir, f"{year}_{month}_{exp_name}_T_{p_level/100}hPa")
                    plot_temperature_maps(exp_name, data_dict['T'].sel(plev=p_level),
                                        combo_dir, p_level)
                    gif_fname = os.path.join(base_output_dir, f"T_{year}_{month}_{exp_name}_{p_level/100}hPa.gif")
                    create_gif_from_directory(combo_dir, gif_fname)
                
                # Compute and plot PV for pressure levels and isentropic surface
                # Reload full pressure data for PV calculation
                if month == 'Feb':
                    data_nocouple_pv, _ = load_year_nocouple_field(year, variables, lat_min=60, lat_max=90)
                    data_small_pv, _ = load_year_small_pert_field(year, variables, lat_min=60, lat_max=90)
                    data_pv = data_nocouple_pv if exp_name == "nocoupl" else data_small_pv
                elif month == 'Mar':
                    _, data_nocouple_pv = load_year_nocouple_field(year, variables, lat_min=60, lat_max=90)
                    _, data_small_pv = load_year_small_pert_field(year, variables, lat_min=60, lat_max=90)
                    data_pv = data_nocouple_pv if exp_name == "nocoupl" else data_small_pv
                
                # PV on pressure levels
                pv_pressure = compute_pv(data_pv, p_levels=pressure_levels)
                for p_level in pressure_levels:
                    combo_dir = os.path.join(base_output_dir, f"{year}_{month}_{exp_name}_PV_{p_level/100}hPa")
                    plot_pv_maps(exp_name, pv_pressure.sel(plev=p_level),
                                combo_dir, p_level=p_level)
                    gif_fname = os.path.join(base_output_dir, f"PV_{year}_{month}_{exp_name}_{p_level/100}hPa.gif")
                    create_gif_from_directory(combo_dir, gif_fname)
                
                # PV on isentropic surface (380K)
                pv_isentropic = compute_pv(data_pv, isentropic_level=isentropic_level)
                combo_dir = os.path.join(base_output_dir, f"{year}_{month}_{exp_name}_PV_{isentropic_level}K")
                plot_pv_maps(exp_name, pv_isentropic, combo_dir, isentropic_level=isentropic_level)
                gif_fname = os.path.join(base_output_dir, f"PV_{year}_{month}_{exp_name}_{isentropic_level}K.gif")
                create_gif_from_directory(combo_dir, gif_fname)
    
    print("\nAll polar vortex GIFs have been generated!")